# Solution 03 — batch producer & partitions

In [ ]:
from confluent_kafka import Producer
from collections import defaultdict, Counter
import json, time, random

BROKER = 'redpanda:29092'

producer = Producer({
    'bootstrap.servers': BROKER,
    'client.id':         'batch-producer',
})

riders = ['rider_3', 'rider_5', 'rider_6', 'rider_7']
orders = {                       # order id -> (customer, restaurant, total in CHF)
    'order-4711': ('Anna',  'Pizzeria Roma',   32.50),
    'order-4712': ('Ben',   'Sushi Bar Kyoto', 47.80),
    'order-4713': ('Clara', 'Burger House',    21.90),
}
STATUSES   = ['placed', 'accepted', 'picked_up', 'delivered']
order_step = {order_id: 0 for order_id in orders}   # how far each order has progressed
topics     = ['rider-locations', 'orders']


def make_event(topic):
    """Return (key, event) — a GPS ping of a random rider or the next status of a random order."""
    if topic == 'rider-locations':
        rider = random.choice(riders)
        return rider, {
            'rider_id':  rider,
            'lat':       round(47.0502 + random.uniform(-0.01, 0.01), 5),
            'lon':       round(8.3093 + random.uniform(-0.01, 0.01), 5),
            'speed_kmh': round(random.uniform(0.0, 35.0), 1),
            'timestamp': time.time(),
        }
    order_id = random.choice(list(orders))
    customer, restaurant, total = orders[order_id]
    status = STATUSES[min(order_step[order_id], len(STATUSES) - 1)]
    order_step[order_id] += 1
    return order_id, {
        'order_id':   order_id,
        'customer':   customer,
        'restaurant': restaurant,
        'total_chf':  total,
        'status':     status,
        'timestamp':  time.time(),
    }

In [ ]:
results = []
def delivery_report(err, msg):
    if not err:
        results.append({'topic': msg.topic(), 'partition': msg.partition(),
                        'key': msg.key().decode()})

for _ in range(15):
    topic      = random.choice(topics)
    key, event = make_event(topic)
    producer.produce(topic, key=key.encode(), value=json.dumps(event).encode(),
                     callback=delivery_report)
producer.flush()

In [ ]:
key_partitions = defaultdict(set)
for r in results:
    key_partitions[(r['topic'], r['key'])].add(r['partition'])

print(f'{"Topic":<15} | {"Key":<10} | Partition(s) | Consistent?')
print('-' * 60)
for (t, k), parts in sorted(key_partitions.items()):
    plist = ', '.join(str(p) for p in sorted(parts))
    flag  = 'yes' if len(parts) == 1 else 'NO!'
    print(f'{t:<15} | {k:<10} | {plist:<12} | {flag}')

## Task A — keyless events

Notice the distribution is *not* uniform — librdkafka uses "sticky" partitioning for keyless messages, for better batching. Across many runs the partitions average out, but in a 6-event burst you usually see one partition dominate. The `rider_id` inside the value plays no role in routing.

In [ ]:
no_key_partitions = []
def no_key_cb(err, msg):
    if not err:
        no_key_partitions.append(msg.partition())
        print(f'  -> P{msg.partition()} offset={msg.offset()}')

for i in range(6):
    event = json.dumps({'rider_id': 'rider_7',
                        'lat': 47.0502 + i * 0.0002, 'lon': 8.3093,
                        'speed_kmh': 15.0, 'timestamp': time.time()})
    producer.produce('rider-locations', key=None, value=event.encode(),
                     callback=no_key_cb)

producer.flush()
print(f'Distribution: {dict(Counter(no_key_partitions))}')